In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.ml.linalg import Vectors, VectorUDT

import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import os

from pyspark.ml.feature import BucketedRandomProjectionLSH

In [ ]:
spark = SparkSession.builder.appName("CarDataAnalysis").getOrCreate()
cars_df = spark.read.csv("/content/drive/MyDrive/ESI TPs/BigData/Data/Lab5/Data/cars_train_data.csv", header=True, inferSchema=True)

cars_df = cars_df.select("Class", "Image")

cars_df.printSchema()
cars_df.show(5)

root
 |-- Class: integer (nullable = true)
 |-- Image: string (nullable = true)

+-----+---------+
|Class|    Image|
+-----+---------+
|   14|00001.jpg|
|    3|00002.jpg|
|   91|00003.jpg|
|  134|00004.jpg|
|  106|00005.jpg|
+-----+---------+
only showing top 5 rows


Q1)- Perform this transformation using the Torchvision package in PyTorch

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize(256),             # Resize the smaller edge to 256 (to ensure we don't distort the aspect ratio of the car.)
    transforms.CenterCrop(224),         # Crop the center to get exactly 224x224
    transforms.ToTensor(),              # Convert image to a PyTorch Tensor (scales the pixel values from (0-255) to (0.0-1.0))
    transforms.Normalize(               # Standardize colors based on ImageNet stats (to ensure that the brightness and contrast of your images match what the model expects.)
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transform:", preprocess)

Transform: Compose(
    Resize(size=256, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


Q2)- Create class C.

In [ ]:
# Load a pre-trained ResNet-18 model
resnet18 = models.resnet18(pretrained=True)

# Set the model to evaluation mode (turns off dropout/batchnorm training)
resnet18.eval()

# Remove the last fully connected layer so we get 512-d from the layer
# before it cause i think that the last layer is the one that give a naming to the images and we just want the vect from the layer before
classC = torch.nn.Sequential(*(list(resnet18.children())[:-1]))



/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 174MB/s]


Q3)- Using class C, calculate the image embeddings for all images.


In [ ]:
import os
import pandas as pd
from PIL import Image
import torch

img_dir = "/content/drive/MyDrive/ESI TPs/BigData/Data/Lab5/Data/cars_train/"

# Convert the Spark DataFrame to a Pandas DataFrame for easy looping
pdf_labels = cars_df.toPandas()

# Q3: Compute embeddings using a simple loop
embeddings_list = []

for index, row in pdf_labels.iterrows():
    img_name = row["Image"]
    img_class = row["Class"]
    img_path = os.path.join(img_dir, img_name)

    # Skip if image doesn't exist
    if not os.path.isfile(img_path):
        continue

    try:
        # Open and preprocess using YOUR variable 'xpreprocess'
        img = Image.open(img_path).convert('RGB')
        img_t = xpreprocess(img)
        batch_t = torch.unsqueeze(img_t, 0)

        # Pass through Class C to get the 512 numbers
        with torch.no_grad():
            output = classC(batch_t)

        # Flatten into a list
        emb = output.squeeze().numpy().tolist()

        # Save to our temporary list
        embeddings_list.append({"Class": img_class, "Image": img_name, "embedding": emb})

    except Exception as e:
        print(f"Error processing {img_name}: {e}")



Q4)- Store the image embeddings in a CSV file.

In [ ]:
# Q4: Store embeddings in a CSV file (Using the 512 Columns trick)
rows_csv = []
for d in embeddings_list:
    row = {"Class": d["Class"], "Image": d["Image"]}
    # Split the 512 numbers into 512 separate columns (e0, e1, e2...)
    for i, v in enumerate(d["embedding"]):
        row[f"e{i}"] = v
    rows_csv.append(row)

# Convert to Pandas and save to your Google Drive Data folder
df_emb = pd.DataFrame(rows_csv)
embeddings_csv_path = "/content/drive/MyDrive/ESI TPs/BigData/Data/Lab5/Data/image_embeddings.csv"
df_emb.to_csv(embeddings_csv_path, index=False)

print(f"✅ Embeddings successfully stored in: {embeddings_csv_path}")

Q5)- Create a Spark DataFrame that contains the image embeddings.

In [ ]:
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.functions import udf, col

# Q5: Create a Spark DataFrame that contains the image embeddings (by loading the CSV)

# 1. Read the CSV file/folder we created in Q4
# Spark will automatically handle reading the part-files inside the folder
df_raw = spark.read.csv("cars_embeddings_output.csv", header=True, inferSchema=True)

# 2. Define a function to parse the text string "[0.1, 0.2, ...]" back into a Vector
def parse_vector(s):
    if s is None:
        return None
    # Remove brackets, split by commas, and convert strings to floats
    values = [float(x) for x in s.strip('[]').split(',')]
    return Vectors.dense(values)

# 3. Register the function as a Spark UDF (User Defined Function)
parse_vector_udf = udf(parse_vector, VectorUDT())

# 4. Create the final DataFrame
# We convert the 'embeddings_str' back to a real 'embeddings' vector column
df_final = df_raw.withColumn("embeddings", parse_vector_udf(col("embeddings_str"))) \
                 .select("Class", "Image", "embeddings")

# Verify the schema to ensure 'embeddings' is a vector type
df_final.printSchema()
df_final.show(5)

root
 |-- Class: integer (nullable = true)
 |-- Image: string (nullable = true)
 |-- embeddings: vector (nullable = true)

+-----+---------+--------------------+
|Class|    Image|          embeddings|
+-----+---------+--------------------+
|   14|00001.jpg|[0.35791105031967...|
|    3|00002.jpg|[1.10152745246887...|
|   91|00003.jpg|[1.30161142349243...|
|  134|00004.jpg|[1.81889152526855...|
|  106|00005.jpg|[0.59427464008331...|
+-----+---------+--------------------+
only showing top 5 rows


In [ ]:
from pyspark.ml.feature import VectorAssembler

# Q6: Transform the relevant columns into a single vector named "features"
# We take our 512-dimensional 'embeddings' and prepare them for the LSH model
assembler = VectorAssembler(
    inputCols=["embeddings"],
    outputCol="features"
)
from pyspark.ml.feature import VectorAssembler

# Q6: Transform the relevant columns into a single vector named "features"
# We take our 512-dimensional 'embeddings' and prepare them for the LSH model
assembler = VectorAssembler(
    inputCols=["embeddings"],
    outputCol="features"
)

# Apply the transformation to our DataFrame
df_assembled = assembler.transform(df_final)

# We can now see the 'features' column at the end of our table
df_assembled.select("Image", "features").show(5, truncate=True)from pyspark.ml.feature import VectorAssembler

# Q6: Transform the relevant columns into a single vector named "features"
# We take our 512-dimensional 'embeddings' and prepare them for the LSH model
assembler = VectorAssembler(
    inputCols=["embeddings"],
    outputCol="features"
)

# Apply the transformation to our DataFrame
df_assembled = assembler.transform(df_final)

# We can now see the 'features' column at the end of our table
df_assembled.select("Image", "features").show(5, truncate=True)from pyspark.ml.feature import VectorAssembler

# Q6: Transform the relevant columns into a single vector named "features"
# We take our 512-dimensional 'embeddings' and prepare them for the LSH model
assembler = VectorAssembler(
    inputCols=["embeddings"],
    outputCol="features"
)

# Apply the transformation to our DataFrame
df_assembled = assembler.transform(df_final)

# We can now see the 'features' column at the end of our table
df_assembled.select("Image", "features").show(5, truncate=True)

# Apply the transformation to our DataFrame
df_assembled = assembler.transform(df_final)

# We can now see the 'features' column at the end of our table
df_assembled.select("Image", "features").show(5, truncate=True)

SyntaxError: invalid syntax (2864290688.py, line 8)